In [1]:
import numpy as np
import pandas as pd
import re
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:

data = pd.read_csv("Dataset/weaksatd/complete.csv")
data['LeadingComment'] = data['LeadingComment'].fillna('').astype(str)
data['Function'] = data['Function'].fillna('').astype(str)

def clean_code_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'(/\*+|\*+/|//|\*)', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['Clean_Comment'] = data['LeadingComment'].apply(clean_code_text)
print("Preprocessing complete!")
print(f"Total samples: {len(data):,}")

Preprocessing complete!
Total samples: 688,134


In [3]:
X_satd = data['Clean_Comment']
y_satd = data['PS']

X_train_satd, X_test_satd, y_train_satd, y_test_satd = train_test_split(
    X_satd, y_satd, test_size=0.20, random_state=42, stratify=y_satd
)
print(f"Train size: {len(X_train_satd):,}, Test size: {len(X_test_satd):,}")
print(f"Train positive %: {y_train_satd.mean()*100:.2f}%, Test positive %: {y_test_satd.mean()*100:.2f}%")

Train size: 550,507, Test size: 137,627
Train positive %: 0.85%, Test positive %: 0.85%


In [4]:
satd_vectorizer = TfidfVectorizer(
    max_features=50000, ngram_range=(1, 2),
    sublinear_tf=True, token_pattern=r'(?u)\b\w+\b'
)
X_train_satd_tfidf = satd_vectorizer.fit_transform(X_train_satd)
X_test_satd_tfidf = satd_vectorizer.transform(X_test_satd)
print(f"Train matrix shape: {X_train_satd_tfidf.shape}")
print(f"Test matrix shape:  {X_test_satd_tfidf.shape}")

Train matrix shape: (550507, 50000)
Test matrix shape:  (137627, 50000)


In [5]:
def evaluate_classifier(name, model, X_test, y_test, target_label_name="Positive (1)"):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc*100:.2f}%  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
    print(f"TN: {tn:,}  FP: {fp:,}  FN: {fn:,}  TP: {tp:,}")
    return {"Model": name, "Accuracy": round(acc,4), "Precision (1)": round(prec,4),
            "Recall (1)": round(rec,4), "F1-Score (1)": round(f1,4)}

In [6]:
satd_results = []

mnb_satd = MultinomialNB()
mnb_satd.fit(X_train_satd_tfidf, y_train_satd)
satd_results.append(evaluate_classifier("Multinomial Naive Bayes", mnb_satd, X_test_satd_tfidf, y_test_satd))

logreg_satd = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)
logreg_satd.fit(X_train_satd_tfidf, y_train_satd)
satd_results.append(evaluate_classifier("Logistic Regression (Balanced)", logreg_satd, X_test_satd_tfidf, y_test_satd))

svm_satd = LinearSVC(class_weight='balanced', dual="auto", random_state=42, max_iter=2000)
svm_satd.fit(X_train_satd_tfidf, y_train_satd)
satd_results.append(evaluate_classifier("Linear SVM (Balanced)", svm_satd, X_test_satd_tfidf, y_test_satd))

df_satd_comparison = pd.DataFrame(satd_results)
print("\n=== SATD (PS) CLASSIFIER COMPARISON TABLE ===")
print(df_satd_comparison.to_string(index=False))


--- Multinomial Naive Bayes ---
Accuracy: 99.15%  Precision: 0.0000  Recall: 0.0000  F1: 0.0000
TN: 136,454  FP: 1  FN: 1,172  TP: 0


c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



--- Logistic Regression (Balanced) ---
Accuracy: 96.78%  Precision: 0.0650  Recall: 0.2073  F1: 0.0990
TN: 132,959  FP: 3,496  FN: 929  TP: 243

--- Linear SVM (Balanced) ---
Accuracy: 98.49%  Precision: 0.1411  Recall: 0.1527  F1: 0.1467
TN: 135,365  FP: 1,090  FN: 993  TP: 179

=== SATD (PS) CLASSIFIER COMPARISON TABLE ===
                         Model  Accuracy  Precision (1)  Recall (1)  F1-Score (1)
       Multinomial Naive Bayes    0.9915         0.0000      0.0000        0.0000
Logistic Regression (Balanced)    0.9678         0.0650      0.2073        0.0990
         Linear SVM (Balanced)    0.9849         0.1411      0.1527        0.1467


In [7]:
X_vuln = data['Function'].apply(clean_code_text)
y_vuln = data['W']

X_train_vuln, X_test_vuln, y_train_vuln, y_test_vuln = train_test_split(
    X_vuln, y_vuln, test_size=0.20, random_state=42, stratify=y_vuln
)
print(f"Train size: {len(X_train_vuln):,}, Test size: {len(X_test_vuln):,}")
print(f"Train positive %: {y_train_vuln.mean()*100:.2f}%, Test positive %: {y_test_vuln.mean()*100:.2f}%")

vuln_vectorizer = TfidfVectorizer(
    max_features=50000, ngram_range=(1, 2),
    sublinear_tf=True, token_pattern=r'(?u)\b\w+\b'
)
X_train_vuln_tfidf = vuln_vectorizer.fit_transform(X_train_vuln)
X_test_vuln_tfidf = vuln_vectorizer.transform(X_test_vuln)
print(f"Train matrix shape: {X_train_vuln_tfidf.shape}")
print(f"Test matrix shape:  {X_test_vuln_tfidf.shape}")

Train size: 550,507, Test size: 137,627
Train positive %: 31.92%, Test positive %: 31.92%
Train matrix shape: (550507, 50000)
Test matrix shape:  (137627, 50000)


In [8]:
vuln_results = []

mnb_vuln = MultinomialNB()
mnb_vuln.fit(X_train_vuln_tfidf, y_train_vuln)
vuln_results.append(evaluate_classifier("Multinomial Naive Bayes", mnb_vuln, X_test_vuln_tfidf, y_test_vuln))

logreg_vuln = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)
logreg_vuln.fit(X_train_vuln_tfidf, y_train_vuln)
vuln_results.append(evaluate_classifier("Logistic Regression (Balanced)", logreg_vuln, X_test_vuln_tfidf, y_test_vuln))

svm_vuln = LinearSVC(class_weight='balanced', dual="auto", random_state=42, max_iter=2000)
svm_vuln.fit(X_train_vuln_tfidf, y_train_vuln)
vuln_results.append(evaluate_classifier("Linear SVM (Balanced)", svm_vuln, X_test_vuln_tfidf, y_test_vuln))

df_vuln_comparison = pd.DataFrame(vuln_results)
print("\n=== VULNERABILITY (W) CLASSIFIER COMPARISON TABLE ===")
print(df_vuln_comparison.to_string(index=False))


--- Multinomial Naive Bayes ---
Accuracy: 79.72%  Precision: 0.7098  Recall: 0.6168  F1: 0.6601
TN: 82,627  FP: 11,075  FN: 16,832  TP: 27,093


c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



--- Logistic Regression (Balanced) ---
Accuracy: 84.05%  Precision: 0.7208  Recall: 0.8164  F1: 0.7656
TN: 79,810  FP: 13,892  FN: 8,066  TP: 35,859

--- Linear SVM (Balanced) ---
Accuracy: 84.85%  Precision: 0.7380  Recall: 0.8142  F1: 0.7743
TN: 81,007  FP: 12,695  FN: 8,160  TP: 35,765

=== VULNERABILITY (W) CLASSIFIER COMPARISON TABLE ===
                         Model  Accuracy  Precision (1)  Recall (1)  F1-Score (1)
       Multinomial Naive Bayes    0.7972         0.7098      0.6168        0.6601
Logistic Regression (Balanced)    0.8405         0.7208      0.8164        0.7656
         Linear SVM (Balanced)    0.8485         0.7380      0.8142        0.7743


In [9]:
X_vuln = data['Function'].apply(clean_code_text)
y_vuln = data['W']

X_train_vuln, X_test_vuln, y_train_vuln, y_test_vuln = train_test_split(
    X_vuln, y_vuln, test_size=0.20, random_state=42, stratify=y_vuln
)
print(f"Train size: {len(X_train_vuln):,}, Test size: {len(X_test_vuln):,}")
print(f"Train positive %: {y_train_vuln.mean()*100:.2f}%, Test positive %: {y_test_vuln.mean()*100:.2f}%")

vuln_vectorizer = TfidfVectorizer(
    max_features=50000, ngram_range=(1, 2),
    sublinear_tf=True, token_pattern=r'(?u)\b\w+\b'
)
X_train_vuln_tfidf = vuln_vectorizer.fit_transform(X_train_vuln)
X_test_vuln_tfidf = vuln_vectorizer.transform(X_test_vuln)
print(f"Train matrix shape: {X_train_vuln_tfidf.shape}")
print(f"Test matrix shape:  {X_test_vuln_tfidf.shape}")

Train size: 550,507, Test size: 137,627
Train positive %: 31.92%, Test positive %: 31.92%
Train matrix shape: (550507, 50000)
Test matrix shape:  (137627, 50000)


In [10]:
vuln_results = []

mnb_vuln = MultinomialNB()
mnb_vuln.fit(X_train_vuln_tfidf, y_train_vuln)
vuln_results.append(evaluate_classifier("Multinomial Naive Bayes", mnb_vuln, X_test_vuln_tfidf, y_test_vuln))

logreg_vuln = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)
logreg_vuln.fit(X_train_vuln_tfidf, y_train_vuln)
vuln_results.append(evaluate_classifier("Logistic Regression (Balanced)", logreg_vuln, X_test_vuln_tfidf, y_test_vuln))

svm_vuln = LinearSVC(class_weight='balanced', dual="auto", random_state=42, max_iter=2000)
svm_vuln.fit(X_train_vuln_tfidf, y_train_vuln)
vuln_results.append(evaluate_classifier("Linear SVM (Balanced)", svm_vuln, X_test_vuln_tfidf, y_test_vuln))

df_vuln_comparison = pd.DataFrame(vuln_results)
print("\n=== VULNERABILITY (W) CLASSIFIER COMPARISON TABLE ===")
print(df_vuln_comparison.to_string(index=False))


--- Multinomial Naive Bayes ---
Accuracy: 79.72%  Precision: 0.7098  Recall: 0.6168  F1: 0.6601
TN: 82,627  FP: 11,075  FN: 16,832  TP: 27,093


c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



--- Logistic Regression (Balanced) ---
Accuracy: 84.05%  Precision: 0.7208  Recall: 0.8164  F1: 0.7656
TN: 79,810  FP: 13,892  FN: 8,066  TP: 35,859

--- Linear SVM (Balanced) ---
Accuracy: 84.85%  Precision: 0.7380  Recall: 0.8142  F1: 0.7743
TN: 81,007  FP: 12,695  FN: 8,160  TP: 35,765

=== VULNERABILITY (W) CLASSIFIER COMPARISON TABLE ===
                         Model  Accuracy  Precision (1)  Recall (1)  F1-Score (1)
       Multinomial Naive Bayes    0.7972         0.7098      0.6168        0.6601
Logistic Regression (Balanced)    0.8405         0.7208      0.8164        0.7656
         Linear SVM (Balanced)    0.8485         0.7380      0.8142        0.7743
